# 세민 다구간 모델 — 행동반응 통일본 (단일경계 방식 이식)

> 3조 후반부 · 2026-06-04 · `세민모델_수정본_통일반영.ipynb` 기반
>
> **프로젝트 목적**: '요금 경계 = 단일구간 vs 다구간'이라는 **단 하나의 변수**만 다르게 두고 최적 정책 차이를 보는 것.
> 따라서 행동반응·목적함수 등 나머지는 **모두 통제(동일)** 해야 결과 차이를 경계 구조 탓으로 해석할 수 있다.

이 노트북은 직전 통일본(수송원가 제외·공통 CPI·탑승당 κ·억원 출력)에 더해 **마지막 두 가지**를 통일한다.

| # | 항목 | 직전 통일본 | 본 통일본 |
|---|---|---|---|
| 2 | **목적함수** | NSB = 복지편익 + 운임수익 | (형태 유지) NSB = 복지편익(수준) + 운임수익 |
| 3 | **행동반응** | 활동보존율 `1-s·b` (탑승량 고정) | **단일경계 ε 방식 이식** — 부담률에 따라 탑승량 자체 감소 |

**핵심 매핑**: 세민의 **부담률 b** 가 단일경계의 **ρ = f/f₀**(부과요금 비율)에 해당한다. 둘 다 "전액요금의 몇 %를 노인이 내는가"로 0~1 범위라 그대로 대응된다.

> **근거 없는 추정치 전면 제거**: 기존 세민 모델의 연령대별 `activity_sensitivity`(활동 민감도)와 `subway_use_rate`(지하철 이용률)는 출처가 불명확한 추정치라 **모두 삭제**했다.
> - 탑승 분배: 지하철이용률 가중을 빼고 **추계인구 비중만** 사용(실데이터, 그룹 간 1인당 탑승 동일 가정).
> - 행동반응: 활동 민감도 대신 **단일경계 모델의 ε(빈도 -0.3, 외출포기 -0.05)** 로 통일.
> - 남는 비추정 입력: 연령 `score`(순서)와 부담률 상·하한(정책 설계값)뿐.


In [1]:
from __future__ import annotations
import csv, math
from dataclasses import dataclass, field
from pathlib import Path
import pandas as pd, numpy as np

# ===== 공통 CPI 시계열 (단일경계 모델과 동일 표) =====
CPI_2020_BASE = {2014:94.196, 2015:94.861, 2016:95.783, 2017:97.645, 2018:99.086,
                 2019:99.466, 2020:100.0, 2021:102.5, 2022:107.72, 2023:111.59,
                 2024:114.18, 2025:117.4}
_CPI_LAST = max(CPI_2020_BASE); FUTURE_INFL = 0.02
def cpi_factor(fr: int, to: int) -> float:
    def g(y):
        if y in CPI_2020_BASE: return CPI_2020_BASE[y]
        if y > _CPI_LAST: return CPI_2020_BASE[_CPI_LAST]*((1+FUTURE_INFL)**(y-_CPI_LAST))
        return CPI_2020_BASE[2014]/((1+FUTURE_INFL)**(2014-y))
    return g(to)/g(fr)

In [2]:
# ===== 데이터 클래스 =====
@dataclass(frozen=True)
class AgeGroup:
    name: str; age_range: str; score: int
    # [통일] 근거 없는 추정치 전면 제거:
    #   - subway_use_rate(연령대별 지하철이용률) 삭제 -> 탑승 분배는 추계인구 비중만(실데이터)
    #   - activity_sensitivity(활동 민감도) 삭제 -> 행동반응은 ε(-0.3, -0.05)로 단일경계와 통일
    #   score는 추정치가 아니라 연령 순서(부담률 로지스틱용)라 유지

@dataclass(frozen=True)
class YearInput:
    year: int; senior_population: int
    group_populations: dict; predicted_free_rides: int

@dataclass(frozen=True)
class PolicyParams:
    base_fare_2026: float = 1550.0
    min_burden: float = 0.0
    max_burden: float = 1.0
    kappa_freq_2014: float = 330.0   # 의료230 + 사회100 (매 탑승)
    kappa_act_2014:  float = 939.0   # 우울322 + 자살617 (외출 자체 포기분)
    # [통일 3] 단일경계 모델 행동반응 파라미터 이식 (활동민감도 방식 폐기)
    discretionary_share: float = 0.62   # sigma: 재량(선택) 통행 비중 (서울연구원 2019)
    epsilon_intensive: float = -0.3     # 빈도 탄력성 (계속 외출하되 횟수 감소)
    epsilon_extensive: float = -0.05    # 외출포기 탄력성 (재량 외출 자체 포기)

@dataclass(frozen=True)
class PolicyConstraints:
    max_burden_by_group: dict; min_burden_by_group: dict

DEFAULT_AGE_GROUPS = [
    AgeGroup("age_80_plus","80세 이상",1),
    AgeGroup("age_75_79","75~79세",2),
    AgeGroup("age_70_74","70~74세",3),
    AgeGroup("age_65_69","65~69세",4),
]
DEFAULT_CONSTRAINTS = PolicyConstraints(
    max_burden_by_group={"age_80_plus":0.10,"age_75_79":0.35,"age_70_74":0.65,"age_65_69":1.00},
    min_burden_by_group={"age_80_plus":0.00,"age_75_79":0.10,"age_70_74":0.30,"age_65_69":0.70},
)

In [3]:
# ===== 연산 로직 =====
def logistic_burden(score, alpha, beta): return 1/(1+math.exp(-(alpha+beta*score)))
def clipped(v, lo, hi): return max(lo, min(hi, v))

def dynamic_age_weights(groups, gpop):
    # [통일] 지하철이용률(추정치) 제거 -> 추계인구 비중으로만 탑승 분배 (그룹 간 1인당 탑승 동일 가정)
    tot = sum(gpop.values())
    return {g.name: gpop[g.name]/tot for g in groups}

def burden_rates_by_group(groups, params, alpha, beta):
    return {g.name: clipped(logistic_burden(g.score,alpha,beta), params.min_burden, params.max_burden) for g in groups}

def satisfies_constraints(brs, cons):
    for g,br in brs.items():
        if br < cons.min_burden_by_group.get(g,0.0) or br > cons.max_burden_by_group.get(g,1.0): return False
    return True

# ===== [통일 3] 행동반응: 단일경계 모델 ε 방식 이식 =====
# 매핑: 세민 '부담률 b' = 단일경계 'rho = f/f0' (둘 다 부과요금 비율, 0~1)
def behavior_split(group_rides, burden_rate, params):
    """그룹 탑승량을 필수(1-sigma)/재량(sigma)으로 나누고, 재량 탑승을 ε로 감소.
       반환: kept(유지 탑승), forgone_dis(줄어든 재량 합), forgone_ext(외출 자체 포기분).
       - 필수 통행: 요금 내도 유지(비탄력) -> 복지손실 0
       - 재량 통행: 외출포기 d_E = |eps_ext|*rho, 빈도배수 m_I = 1 + eps_int*rho"""
    rho = burden_rate
    sigma = params.discretionary_share
    trips_disc = group_rides * sigma
    d_E = clipped(abs(params.epsilon_extensive) * rho, 0.0, 1.0)   # 외출 포기율
    m_I = max(0.0, 1.0 + params.epsilon_intensive * rho)           # 빈도 배수
    forgone_ext = trips_disc * d_E                                 # 외출 포기로 사라진 재량 탑승
    forgone_int = trips_disc * (1.0 - d_E) * (1.0 - m_I)           # 빈도 감소로 사라진 재량 탑승
    forgone_dis = forgone_ext + forgone_int
    kept = group_rides - forgone_dis                               # 유지 탑승 = 필수 + 잔류 재량
    return kept, forgone_dis, forgone_ext

def evaluate_policy(yi: YearInput, groups, params: PolicyParams, alpha, beta):
    y = yi.year
    fare = params.base_fare_2026 * cpi_factor(2026, y)
    kf = params.kappa_freq_2014 * cpi_factor(2014, y)   # 의료+사회 (매 탑승)
    ka = params.kappa_act_2014  * cpi_factor(2014, y)   # 우울+자살 (외출포기분)
    rw = dynamic_age_weights(groups, yi.group_populations)
    brs = burden_rates_by_group(groups, params, alpha, beta)
    rows = []
    for g in groups:
        b = brs[g.name]
        group_rides = yi.predicted_free_rides * rw[g.name]
        kept, forgone_dis, forgone_ext = behavior_split(group_rides, b, params)

        # 운임수익 = 유지 탑승 x (요금 x 부담률)   [단일경계: kept x f]
        fare_revenue = kept * fare * b
        # 복지편익(수준) = 현행무임 baseline - 손실   [단일경계 모델과 동일 회계]
        welfare_baseline = group_rides * kf
        welfare_loss = forgone_dis * kf + forgone_ext * ka
        welfare_total = welfare_baseline - welfare_loss
        # 목적함수(세민 가법형): NSB = 복지편익 + 운임수익
        net_social_benefit = welfare_total + fare_revenue

        rows.append({"year":y,"group":g.name,"age_range":g.age_range,
                     "burden_rate":b,
                     "trip_reduction":(forgone_dis/group_rides if group_rides else 0.0),
                     "kept_rides":round(kept),"forgone_dis":round(forgone_dis),
                     "predicted_rides":round(group_rides),"fare":fare,
                     "welfare_total":welfare_total,"fare_revenue":fare_revenue,
                     "net_social_benefit":net_social_benefit,"alpha":alpha,"beta":beta})
    return rows

def summarize(rows):
    return {"year":int(rows[0]["year"]),
            "welfare_total":sum(r["welfare_total"] for r in rows),
            "fare_revenue":sum(r["fare_revenue"] for r in rows),
            "net_social_benefit":sum(r["net_social_benefit"] for r in rows),
            "alpha":float(rows[0]["alpha"]),"beta":float(rows[0]["beta"])}

def frange(a,b,s):
    c=a
    while c<=b+1e-9: yield round(c,10); c+=s

def optimize_policy(year_inputs, groups, params, cons=DEFAULT_CONSTRAINTS):
    details, summaries = [], []
    alphas, betas = list(frange(-5.0,1.0,0.25)), list(frange(0.25,2.5,0.25))
    for yi in year_inputs:
        best_d, best_s = None, None
        for a in alphas:
            for b in betas:
                if not satisfies_constraints(burden_rates_by_group(groups,params,a,b), cons): continue
                cd = evaluate_policy(yi, groups, params, a, b); cs = summarize(cd)
                if best_s is None or cs["net_social_benefit"] > best_s["net_social_benefit"]:
                    best_d, best_s = cd, cs
        if best_d is None: raise RuntimeError(f"{yi.year}: 실현 가능한 정책 없음")
        details.extend(best_d); summaries.append(best_s)
    return details, summaries

In [4]:
# ===== 억원/연 포맷 출력 =====
def print_report_eok(year, summaries, details):
    s = next((r for r in summaries if r["year"]==year), None)
    if not s: print(f"{year}년 데이터 없음"); return
    print(f"\n===== {year}년 연령대별 최적 정책 (행동반응 통일본) =====")
    print(f"  복지편익(수준):  {s['welfare_total']/1e8:>10,.0f} 억원/연")
    print(f"  운임수익:        {s['fare_revenue']/1e8:>10,.0f} 억원/연")
    print(f"  순사회편익(NSB): {s['net_social_benefit']/1e8:>10,.0f} 억원/연  (= 복지편익 + 운임수익)")
    print(f"  최적 alpha/beta: {s['alpha']:.2f} / {s['beta']:.2f}")
    print(f"  {'연령대':<10}{'부담률':>8}{'탑승감소':>9}")
    for r in [d for d in details if d['year']==year]:
        print(f"  {r['age_range']:<10}{r['burden_rate']*100:>7.1f}%{r['trip_reduction']*100:>8.1f}%")

print("행동반응 통일본 모듈 로드 완료.")

행동반응 통일본 모듈 로드 완료.


In [5]:
# ===== 데이터 로더 (각세별 추계 -> 4개 연령빈) =====
from pathlib import Path
import pandas as pd, warnings
warnings.filterwarnings("ignore")

# 프로젝트 루트 자동 탐색 (dataset 폴더가 있는 상위 디렉터리)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "dataset").exists():
    ROOT = ROOT.parent
RIDES = ROOT / "dataset" / "processed" / "서울시_2026_2050_무임승차_노인수요_최종_병철v2.csv"
POP   = ROOT / "dataset" / "팀 데이터" / "인구추이(추계인구_각세별)_2026_2050.xlsx"

def load_group_populations(xlsx_path):
    """각세별 추계인구 xlsx -> {연도: {연령빈: 인구}} (총인구 행, 65세 이상 4구간)."""
    df = pd.read_excel(xlsx_path, header=2)
    df = df[df["구분별(1)"] == "총인구"].copy()
    df["시점"] = pd.to_numeric(df["시점"], errors="coerce")
    df = df.dropna(subset=["시점"]); df["시점"] = df["시점"].astype(int)
    bins = {"age_65_69":   [f"{i}세" for i in range(65, 70)],
            "age_70_74":   [f"{i}세" for i in range(70, 75)],
            "age_75_79":   [f"{i}세" for i in range(75, 80)],
            "age_80_plus": [f"{i}세" for i in range(80, 100)] + ["100세 이상", "100세이상"]}
    def to_num(s):
        return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False)
                             .str.replace("-", "0", regex=False), errors="coerce").fillna(0)
    for name, cols in bins.items():
        cc = [c for c in cols if c in df.columns]; df[name] = sum(to_num(df[c]) for c in cc)
    return {int(r["시점"]): {n: int(r[n]) for n in bins} for _, r in df.iterrows()}

def load_rides(csv_path):
    """병철 최종예측 CSV -> {연도: 연간 노인 무임 탑승량}."""
    df = pd.read_csv(csv_path)
    col = "final_pred" if "final_pred" in df.columns else "연간_총_최종수요"
    return {int(r["year"]): int(round(float(r[col]))) for _, r in df.iterrows()}

def generate_year_inputs(pop_path=POP, rides_path=RIDES):
    gp_map = load_group_populations(pop_path); rides_map = load_rides(rides_path)
    return [YearInput(y, sum(gp_map[y].values()), gp_map[y], rides_map[y])
            for y in range(2026, 2051) if y in gp_map and y in rides_map]

In [6]:
# ===== 2026~2050 실행 =====
year_inputs = generate_year_inputs()
print(f"연도입력 {len(year_inputs)}개 로드 ({year_inputs[0].year}~{year_inputs[-1].year})")
details, summaries = optimize_policy(year_inputs, DEFAULT_AGE_GROUPS, PolicyParams())
for y in (2026, 2035, 2050):
    print_report_eok(y, summaries, details)

연도입력 25개 로드 (2026~2050)

===== 2026년 연령대별 최적 정책 (행동반응 통일본) =====
  복지편익(수준):         964 억원/연
  운임수익:             1,898 억원/연
  순사회편익(NSB):      2,861 억원/연  (= 복지편익 + 운임수익)
  최적 alpha/beta: -4.00 / 1.50
  연령대            부담률     탑승감소
  80세 이상        7.6%     1.6%
  75~79세       26.9%     5.8%
  70~74세       62.2%    13.1%
  65~69세       88.1%    18.4%

===== 2035년 연령대별 최적 정책 (행동반응 통일본) =====
  복지편익(수준):       1,472 억원/연
  운임수익:             2,500 억원/연
  순사회편익(NSB):      3,972 억원/연  (= 복지편익 + 운임수익)
  최적 alpha/beta: -4.00 / 1.50
  연령대            부담률     탑승감소
  80세 이상        7.6%     1.6%
  75~79세       26.9%     5.8%
  70~74세       62.2%    13.1%
  65~69세       88.1%    18.4%

===== 2050년 연령대별 최적 정책 (행동반응 통일본) =====
  복지편익(수준):       2,958 억원/연
  운임수익:             3,994 억원/연
  순사회편익(NSB):      6,952 억원/연  (= 복지편익 + 운임수익)
  최적 alpha/beta: -4.00 / 1.50
  연령대            부담률     탑승감소
  80세 이상        7.6%     1.6%
  75~79세       26.9%     5.8%
  70~74세       62.2%    13.1%
  65~69세       88.1%  

---
### 통일 결과 — 두 모델이 이제 공유하는 것

본 통일본 적용 후, 단일경계 모델과 세민 다구간 모델은 **다음을 모두 동일하게** 사용한다.

| 구성 | 공통 내용 |
|---|---|
| CPI | 통계청 2020=100, 2026후 연 2.0% |
| 복지편익 단가 | 탑승당 KOTI κ (κ_freq=330, κ_act=939, 2014→CPI) |
| 수송원가 | 제외 (한계비용 ≈ 0) |
| **행동반응** | **필수/재량 분해 + ε(빈도 -0.3, 외출포기 -0.05)** |
| **목적함수** | **NSB = 복지편익(baseline - 손실) + 운임수익** (가법 수준형) |
| 출력 단위 | 억원/연 |

> 이제 **유일한 차이는 '요금 경계 구조'(단일 threshold vs 연령대별 다구간 부담률)** 뿐 → 결과(NSB) 차이를 경계 구조의 효과로 해석할 수 있다.

### 단일경계 모델 쪽 (참고)
`nsw_simulator_v3.py`는 이미 `social_welfare_eok_yearly = welfare_total + revenue` (가법 수준형, 주석 "세민 모델과 동일 정의")를 계산해 둔다. 따라서 단일경계 모델은 **수식 변경 없이**, 보고/최적화 기준만 `delta_nsw`(ΔW) -> `social_welfare`(W)로 바꾸면 본 통일본과 동일한 목적함수가 된다. (ΔW = W - 상수[현행무임 baseline] 이라 최적 정책은 동일)

### 남은 작업
- **데이터 로더**(`generate_year_inputs`, `_read_data_safe`)와 `main()`은 원본과 동일하므로 원본 셀을 붙여 실행 (인구추이 xlsx + 무임예측 CSV 경로 동일).
- 실행 후 두 모델 NSB를 같은 기준으로 비교 표/그래프화.
